# Gate A1 — inspectable data audit

## TL;DR

This notebook is a reader-facing view of the executable Gate A1 evidence. It does not create an alternative split or feature set: all tables below come from the machine report and frozen manifests produced by `scripts/run_gate_a1.py`.

The expected decision is `PASS_WITH_WARNINGS`: no critical contract/leakage failures, T1 ready for baseline development through manifests, T5 technically frozen but statistically sparse, and test sealed until a final candidate is frozen.

## Context and methods

Canonical T1 sources are the next-planned feature/target tables plus the formal feature and target contracts. The audit checks grain, uniqueness, temporal split boundaries, leakage, train-only preprocessing, test access, missingness, drift, join coverage, and three dependency-aware validation designs. Historical `next_cycle_*` tables are comparison-only.

In [1]:
from pathlib import Path
import json
import pandas as pd

root = Path.cwd().resolve()
while not (root / 'pyproject.toml').is_file():
    if root.parent == root:
        raise RuntimeError('Repository root not found')
    root = root.parent

artifact_dir = root / 'artifacts' / 'data_quality'
report = json.loads((artifact_dir / 'gate_a1_report.json').read_text(encoding='utf-8'))
split_summary = pd.read_csv(artifact_dir / 'split_summary.csv')
checks = pd.read_csv(artifact_dir / 'duplicate_and_grain_checks.csv')
findings = pd.read_csv(artifact_dir / 'gate_a1_findings.csv')
membership = pd.read_csv(artifact_dir / 'membership_inconsistency_mapping.csv')
missingness = pd.read_csv(artifact_dir / 'feature_missingness_by_split.csv')
drift = pd.read_csv(artifact_dir / 'drift_summary.csv')
validation_design = pd.read_csv(artifact_dir / 'validation_design_summary.csv')

pd.set_option('display.max_colwidth', 100)
{
    'gate': report['gate'],
    'status': report['status'],
    'critical_failures': report['summary']['critical_failures'],
    'checks': report['summary']['checks'],
}

{'gate': 'A1_DATA_QUALITY',
 'status': 'PASS_WITH_WARNINGS',
 'critical_failures': 0,
 'checks': 31}

## Data: governed inputs and frozen manifests

In [2]:
pd.DataFrame([
    {'role': 'canonical', 'name': name, **meta}
    for name, meta in report['canonical_inputs'].items()
] + [
    {'role': 'historical_only', 'name': name, **meta}
    for name, meta in report['historical_only_inputs'].items()
])[['role', 'name', 'path', 'sha256']]

,role,name,path,sha256
0,canonical,early_warning_labels,SKRU1_ACTUAL_DATA_TABLES_v1/02_eda_targets_v1/target_tables/early_warning_labels_formal.csv,6eb0c504d8bda621cfb82d59baefc8614c12fd3bec6561b9fa23a01daad30bc7
1,canonical,feature_contract,SKRU1_ACTUAL_DATA_TABLES_v1/02_eda_targets_v1/target_tables/formal_feature_contract.csv,241bad48e67659ffc953349cd2bce59e61f2a2867d57698abd6574edd4a05272
2,canonical,features,SKRU1_ACTUAL_DATA_TABLES_v1/02_eda_targets_v1/target_tables/next_planned_features.csv,5d360910e8eeea10a2572bb35d3de4f0b9b316ec5a6ee3b43389c07b09848ab6
3,canonical,operational_targets,SKRU1_ACTUAL_DATA_TABLES_v1/02_eda_targets_v1/target_tables/next_planned_operational_targets.csv,289912a0b2c79ee1163529c6998d879594062cf0d453ec7665b2bb06f562ee1b
4,canonical,target_contract,SKRU1_ACTUAL_DATA_TABLES_v1/02_eda_targets_v1/target_tables/target_contract.json,c679a6b83d26675d50b59de57cb3c12cdeacc6dd7c2e178364e434ed25a526e8
5,historical_comparison_only,features,SKRU1_ACTUAL_DATA_TABLES_v1/01_reconstruction_v3_2/model_ready/next_cycle_features.csv,bd1eed0554b56a0daa2df725a06cbc9d77d07c1abfd3cc156bbee99bf60f2a58
6,historical_comparison_only,targets,SKRU1_ACTUAL_DATA_TABLES_v1/01_reconstruction_v3_2/evaluation_only/next_cycle_targets.csv,9e9d2d2f0b8e186d88ca2519b15d7ae841b759ea4c301d7c3359ca3ed6370c4c


In [3]:
split_summary[[
    'task', 'version', 'split', 'rows',
    'current_date_min', 'current_date_max',
    'target_date_min', 'target_date_max',
    'points', 'profiles', 'missing_feature_fraction',
    'positive', 'negative', 'censored', 'sample_ids_sha256'
]]

,task,version,split,rows,current_date_min,current_date_max,target_date_min,target_date_max,points,profiles,missing_feature_fraction,positive,negative,censored,sample_ids_sha256
0,T1,t1_v1,train,911,2018-10-16,2023-07-25,2019-02-12,2023-11-07,98,14,0.032382,0,0,0,2cd9ef9d27536d6d4e63ef0a56ad51002823d1306ff7c39ac6538586596b6a8c
1,T1,t1_v1,validation,130,2023-11-07,2024-05-14,2024-01-30,2024-09-03,90,14,0.034154,0,0,0,f72ce0a50c16bccca3ca87df4f4a7465b4d9eca2933e0dec9b3bd39e7336a62e
2,T1,t1_v1,test,175,2024-05-14,2025-08-26,2025-07-22,2025-11-04,89,14,0.033486,0,0,0,8d0db55dba8db96297c6b816e65e2601fdc4bbc014a81b298241b632bcf9a66a
3,T5,t5_v1,train,942,2018-10-16,2023-05-16,2019-02-12,2023-11-07,98,14,0.032569,12,930,0,8533a2a79b340ccefa8762408391fd05e4a18cb39b32d017cc970332853f614f
4,T5,t5_v1,validation,211,2023-07-25,2024-05-14,2023-11-07,2025-07-22,96,14,0.033649,4,207,0,b17b2accb3e53d1a00e2a828fb3d6a9ceb3af2aa9fec3b8a07ce5130b235b9c1
5,T5,t5_v1,test_complete,28,2024-07-09,2024-09-03,2025-07-22,2025-07-22,28,5,0.037143,1,27,0,9ecd7202dc5d1fcbb05b8d6328d46b1bdb6dba1803682407f3874ec093c209b4
6,T5,t5_v1,test_censored,93,2025-07-22,2025-08-26,2025-08-26,2025-11-04,84,14,0.034194,0,0,93,3ff9331474211166bbe9b86166572d53d89a0e1b73d1d46b4b8f6f281ca455dc


## Results: executable checks and dependency structure

In [4]:
checks.groupby(['severity', 'status'], as_index=False).size().sort_values(['severity', 'status'])

,severity,status,size
0,critical,PASS,30
1,high,FAIL,1


In [5]:
checks.loc[checks['status'].ne('PASS'), ['check_id', 'severity', 'dimension', 'observed', 'expected', 'details']]

,check_id,severity,dimension,observed,expected,details
27,A1-ID-004,high,identifier_semantics,58,0,sample_id is unique but its final token still reflects the historical next-available target in t...


In [6]:
report['grain']

{'current_dates_shared_by_multiple_profiles': 26,
 'origin_rows': 1274,
 'origins_per_point_max': 18,
 'origins_per_point_median': 12.0,
 'origins_per_point_min': 8,
 'point_trajectories': 98,
 'points_repeated_over_time': 98,
 'profiles': 14,
 'unique_sample_ids': 1274}

### Membership reconciliation

The row-level mapping below proves why 18 source membership inconsistencies affect only 6 model origins. It distinguishes mapped WORK targets, REF rows outside the model universe, and WORK rows with no eligible prior origin.

In [7]:
membership.groupby(['mapping_reason', 'maps_to_unlabeled_origin'], as_index=False).size()

,mapping_reason,maps_to_unlabeled_origin,size
0,maps_to_unlabeled_model_origin,True,6
1,no_eligible_prior_origin_in_canonical_candidate_frame,False,8
2,reference_point_outside_WORK_model_universe,False,4


In [8]:
membership[[
    'campaign_id', 'point_id', 'profile_id', 'point_type',
    'target_origin_sample_id', 'label_status', 'mapping_reason'
]]

,campaign_id,point_id,profile_id,point_type,target_origin_sample_id,label_status,mapping_reason
0,C005,P-H04-REF-A,P-H04,REF,NaN,NaN,reference_point_outside_WORK_model_universe
1,C005,P-H04-REF-B,P-H04,REF,NaN,NaN,reference_point_outside_WORK_model_universe
2,C005,P-H04-W001,P-H04,WORK,NaN,NaN,no_eligible_prior_origin_in_canonical_candidate_frame
3,C005,P-H04-W002,P-H04,WORK,NaN,NaN,no_eligible_prior_origin_in_canonical_candidate_frame
4,C005,P-H04-W003,P-H04,WORK,NaN,NaN,no_eligible_prior_origin_in_canonical_candidate_frame
5,C005,P-H04-W004,P-H04,WORK,NaN,NaN,no_eligible_prior_origin_in_canonical_candidate_frame
6,C005,P-H04-W005,P-H04,WORK,NaN,NaN,no_eligible_prior_origin_in_canonical_candidate_frame
7,C005,P-H04-W006,P-H04,WORK,NaN,NaN,no_eligible_prior_origin_in_canonical_candidate_frame
8,C005,P-H04-W007,P-H04,WORK,NaN,NaN,no_eligible_prior_origin_in_canonical_candidate_frame
9,C010,P-H07-REF-A,P-H07,REF,NaN,NaN,reference_point_outside_WORK_model_universe


### Missingness and temporal drift

In [9]:
missingness.sort_values('missing_fraction', ascending=False).head(20)[[
    'task', 'split', 'feature', 'missing_count', 'missing_fraction'
]]

,task,split,feature,missing_count,missing_fraction
88,T1,validation,lithology__standard_uncertainty,130,1.000000
38,T1,train,lithology__standard_uncertainty,911,1.000000
138,T1,test,lithology__standard_uncertainty,175,1.000000
188,T5,train,lithology__standard_uncertainty,942,1.000000
238,T5,validation,lithology__standard_uncertainty,211,1.000000
288,T5,test_complete,lithology__standard_uncertainty,28,1.000000
338,T5,test_censored,lithology__standard_uncertainty,93,1.000000
291,T5,test_complete,terrain_TRI_relative,12,0.428571
295,T5,test_complete,terrain_roughness_relative,12,0.428571
341,T5,test_censored,terrain_TRI_relative,33,0.354839


In [10]:
drift.loc[
    drift['feature_type'].eq('numeric') & drift['comparison_split'].str.startswith('test')
].sort_values('absolute_smd', ascending=False).head(20)[[
    'task', 'comparison_split', 'feature', 'absolute_smd',
    'missing_fraction_reference', 'missing_fraction_comparison'
]]

,task,comparison_split,feature,absolute_smd,missing_fraction_reference,missing_fraction_comparison
158,T5,test_complete,forecast_horizon_days,4.370064,0.0,0.0
246,T5,test_censored,terrain_roughness_relative__standard_uncertainty,4.000000,0.0,0.0
242,T5,test_censored,terrain_TRI_relative__standard_uncertainty,4.000000,0.0,0.0
163,T5,test_complete,profile_rate_std_mm_y,3.788039,0.0,0.0
207,T5,test_censored,days_since_previous_observation,3.294866,0.0,0.0
150,T5,test_complete,n_history,3.075516,0.0,0.0
200,T5,test_censored,n_history,2.626490,0.0,0.0
162,T5,test_complete,profile_mean_rate_mm_y,2.381629,0.0,0.0
50,T1,test,n_history,2.319875,0.0,0.0
58,T1,test,forecast_horizon_days,1.895940,0.0,0.0


### Dependency-aware validation designs

In [11]:
validation_design.groupby('design').agg(
    folds=('fold_id', 'nunique'),
    min_train_rows=('train_rows', 'min'),
    max_train_rows=('train_rows', 'max'),
    min_validation_rows=('validation_rows', 'min'),
    max_validation_rows=('validation_rows', 'max'),
)

,folds,min_train_rows,max_train_rows,min_validation_rows,max_validation_rows
design,,,,,
leave_profile_out,14,942,990,51,99
leave_zone_out,4,545,951,90,496
rolling_origin,5,823,1027,14,88


## Findings

In [12]:
findings[['finding_id', 'severity', 'status', 'evidence', 'impact', 'remediation']]

,finding_id,severity,status,evidence,impact,remediation
0,A1-F-001,high,CONTROLLED,1274 origins collapse to 98 point trajectories and 14 profiles.,Row-wise uncertainty estimates would be overconfident and random splits would leak trajectory id...,"Use temporal manifests, rolling origin, leave-profile-out, and leave-zone-out only."
1,A1-F-002,high,CONTROLLED,18 inconsistent membership rows map to 6 unlabeled origins; 4 are REF and 8 WORK rows have no el...,The previously disconnected counts 18 and 6 can now be audited row by row.,Keep the 6 origins out of loss; patch membership status in the next source-data revision.
2,A1-F-003,high,OPEN,58 sample_id values encode a historical target token different from canonical target_campaign_id.,Humans or code that parses sample_id may silently recover the wrong target semantics.,Treat sample_id as an opaque join key in v1; regenerate IDs and version all manifests in a futur...
3,A1-F-004,medium,OPEN,terrain_TRI_relative missingness=32.418%; lithology uncertainty missingness=100.000%.,Imputation and missingness indicators will materially affect several feature families.,"Fit imputation on train only, retain missing indicators, and report ablations for sparse terrain..."
4,A1-F-005,high,OPEN,T5 has 17 complete positive labels in total and only 1 in test_complete.,Headline classification metrics will have very high variance and threshold tuning is fragile.,"Use average precision, fixed-FPR recall, confidence intervals, and treat T5 conclusions as explo..."
5,A1-F-006,high,OPEN,Largest T1 train-to-test numeric drift is n_history with |SMD|=2.320.,Temporal generalization is harder than within-period validation and must drive model selection.,Report drift-aware temporal results and avoid random resampling.
6,A1-F-007,medium,CONTROLLED,No authoritative zone_id exists; spatial_quadrants_v1 freezes coordinate-median quadrants for sp...,The proxy supports reproducible spatial OOD checks but is not an engineering zoning claim.,Replace with a domain-governed zone map in a new split version when available.
7,A1-F-008,medium,CONTROLLED,The model-facing loader seals test until a matching frozen-candidate record is present; raw sour...,"Accidental model/test coupling is blocked in project APIs, but deliberate direct file access is ...",Run models through skru1.splits.load_split_dataset and retain source-code scanning in CI.


## Takeaways

1. T1 can proceed to controlled baselines, but only through `skru1.splits.load_split_dataset`; test remains sealed.
2. Random row splitting is invalid because 1,274 origins represent only 98 repeated point trajectories inside 14 profiles.
3. The 18-versus-6 discrepancy is reconciled and no longer an unexplained inconsistency.
4. T5 has too few positives for strong safety claims; use it as an exploratory secondary task with uncertainty-aware reporting.
5. The leave-zone-out v1 groups are geometric proxy quadrants, not authoritative engineering zones.
6. `sample_id` must be treated as opaque in v1 because 58 IDs retain a historical target token.